# DataVint Quick Start (5 minutes)

The fastest way to get started with DataVint data quality checks.

In [2]:
# Setup
import sys
sys.path.insert(0, '..')

import datavint as dv

## 1. Profile Your Dataset (< 1 second)

Get a quick overview before running quality checks.

In [3]:
dv.profile_dataset(
    "../playground/raw_data/movielens_train.csv",
    label_col="label"
)


═══════════════════════════════════════════════════════════════
📊 Dataset Profile
═══════════════════════════════════════════════════════════════
📁 Source: movielens_train.csv
📏 Shape: 80,668 rows × 8 columns
💾 Memory: 9.2 MB

📋 Column Types:
   • Numeric: 6 columns
     userId, movieId, rating, year, month, ... (+1 more)
   • Categorical: 1 columns
     genre

⚠️  Missing Values:
   • No missing values ✅

🎯 Label Distribution (label):
   • Positive (1): 48.7% (39,254 samples)
   • Negative (0): 51.3% (41,414 samples)
   • Balance: Excellent ✅ (perfectly balanced)

📝 Sample Preview (first 5 rows):
 userId  movieId  label  rating     genre  year  month  user_activity
    429      595      1     5.0 Animation  1996      3             58
    429      588      1     5.0 Adventure  1996      3             58
    429      590      1     5.0 Adventure  1996      3             58
    429      592      1     5.0    Action  1996      3             58
    429      432      0     3.0 Adventure  1

## 2. Compare Train vs Test (< 1 second)

Check for distribution shifts before training.

In [4]:
dv.compare_datasets(
    train_data="../playground/raw_data/movielens_train.csv",
    test_data="../playground/raw_data/movielens_test.csv",
    label_col="label"
)


═══════════════════════════════════════════════════════════════
📊 Dataset Comparison: Train vs Test
═══════════════════════════════════════════════════════════════

                               Train            Test          Δ
───────────────────────────────────────────────────────────────
Rows                          80,668          20,168     -75.0%
Columns                            8               8          0
Memory                          9.2 MB            2.3 MB     -75.0%
Missing %                       0.0%            0.0%      +0.0%
Duplicates %                    0.0%            0.0%      +0.0%

Label (label):
  0                            51.3%           53.8%      +2.4%
  1                            48.7%           46.2%      -2.4%

⚠️  Large difference in row count (>50%)

Next: Run dv.detect_issues() with serving_statistics for detailed skew analysis


## 3. Generate Statistics (2-5 seconds)

Compute detailed per-feature statistics.

In [5]:
train_stats = dv.generate_statistics(
    "../playground/raw_data/movielens_train.csv",
    label_col="label"
)

test_stats = dv.generate_statistics(
    "../playground/raw_data/movielens_test.csv",
    label_col="label"
)

print(f"✅ Train: {train_stats.n_rows:,} rows")
print(f"✅ Test:  {test_stats.n_rows:,} rows")

[datavint] Loading data from ../playground/raw_data/movielens_train.csv
[datavint] Computing statistics for 80,668 rows, 8 columns
[datavint] Generated statistics for 7 features
[datavint] Loading data from ../playground/raw_data/movielens_test.csv
[datavint] Computing statistics for 20,168 rows, 8 columns
[datavint] Generated statistics for 7 features


✅ Train: 80,668 rows
✅ Test:  20,168 rows


## 4. Detect Quality Issues (< 1 second)

Run all 6 detectors automatically.

In [6]:
issues = dv.detect_issues(
    statistics=train_stats,
    serving_statistics=test_stats  # Optional: enables train-test skew detection
)

dv.display_issues(issues)


8 issue(s) detected:

🔴 [out_of_range] userId
   Values outside training range: max=610.00 > train_max=609.00 (1.00 above, 0%)
   Direction: NE↑ AUC↓  Severity: HIGH

🔴 [out_of_range] movieId
   Values outside training range: max=193609.00 > train_max=155168.00 (38441.00 above, 25%)
   Direction: NE↑ AUC↓  Severity: HIGH

🔴 [out_of_range] year
   Values outside training range: max=2018.00 > train_max=2016.00 (2.00 above, 0%)
   Direction: NE↑ AUC↓  Severity: HIGH

🔴 [train_test_skew] year
   Distribution differs between train and test (JS divergence: 0.739)
   Direction: NE↑ AUC↓  Severity: HIGH

🔴 [train_test_skew] movieId
   Distribution differs between train and test (JS divergence: 0.657)
   Direction: NE↑ AUC↓  Severity: HIGH

🟡 [train_test_skew] user_activity
   Distribution differs between train and test (JS divergence: 0.153)
   Direction: NE↑ AUC↓  Severity: MEDIUM

🟡 [train_test_skew] month
   Distribution differs between train and test (JS divergence: 0.123)
   Direction: N

## 5. Now Try With Anomalous Data

See what DataVint detects in a dataset with injected quality issues.

In [7]:
# Profile first
dv.profile_dataset(
    "../playground/raw_data/movielens_anomalous.csv",
    label_col="label"
)


═══════════════════════════════════════════════════════════════
📊 Dataset Profile
═══════════════════════════════════════════════════════════════
📁 Source: movielens_anomalous.csv
📏 Shape: 81,474 rows × 8 columns
💾 Memory: 9.1 MB

📋 Column Types:
   • Numeric: 6 columns
     userId, movieId, rating, year, month, ... (+1 more)
   • Categorical: 1 columns
     genre

⚠️  Missing Values:
   • Total: 4,082 missing cells (0.6%)
     - genre: 4,082 (5.0%)

🎯 Label Distribution (label):
   • Positive (1): 48.7% (39,670 samples)
   • Negative (0): 51.3% (41,804 samples)
   • Balance: Excellent ✅ (perfectly balanced)

📝 Sample Preview (first 5 rows):
 userId  movieId  label  rating     genre  year  month  user_activity
    429      595      1     5.0 Animation  1996      3           1211
    429      588      1     5.0 Adventure  1996      3             58
    429      590      1     5.0 Adventure  1996      3             58
    429      592      1     5.0    Action  1996      3             58

In [8]:
# Detect issues
anomalous_stats = dv.generate_statistics(
    "../playground/raw_data/movielens_anomalous.csv",
    label_col="label"
)

anomalous_issues = dv.detect_issues(anomalous_stats)

dv.display_issues(anomalous_issues)

[datavint] Loading data from ../playground/raw_data/movielens_anomalous.csv
[datavint] Computing statistics for 81,474 rows, 8 columns
[datavint] Generated statistics for 7 features


✅ No issues detected!


## That's It!

You now know the complete DataVint workflow:

1. **Profile** → Quick dataset overview
2. **Compare** → Check train/test similarity
3. **Statistics** → Detailed feature analysis
4. **Detect** → Find quality issues
5. **Fix** → Apply manifest (coming in v0.2)

**Next steps:**
- Try `data_profiling_demo.ipynb` for detailed examples
- Use DataVint on your own datasets
- Check `docs/` for API documentation